# Azure RAG with LlamaIndex
Upload · Index · Query · Delete · Rebuild

## Step 1 — Install

In [ ]:
!pip install llama-index llama-index-llms-azure-openai llama-index-embeddings-azure-openai python-dotenv -q

## Step 2 — Imports & Config

Create a `.env` file:
```
AZURE_OPENAI_API_KEY=...
AZURE_OPENAI_ENDPOINT=https://your-resource.openai.azure.com/
AZURE_OPENAI_API_VERSION=2024-02-01
AZURE_OPENAI_DEPLOYMENT=gpt-4o
AZURE_OPENAI_EMBEDDING_DEPLOYMENT=text-embedding-3-small
```

In [ ]:
import os, shutil
from pathlib import Path
from dotenv import load_dotenv
from llama_index.core import Settings, VectorStoreIndex, SimpleDirectoryReader, StorageContext, load_index_from_storage
from llama_index.llms.azure_openai import AzureOpenAI
from llama_index.embeddings.azure_openai import AzureOpenAIEmbedding

load_dotenv()

DATA_DIR = Path("../data/qna/")
STORAGE_DIR = Path("./storage")
DATA_DIR.mkdir(exist_ok=True)
STORAGE_DIR.mkdir(exist_ok=True)

Settings.llm = AzureOpenAI(
    model="gpt-4o",
    deployment_name=os.getenv("AZURE_OPENAI_DEPLOYMENT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
)

Settings.embed_model = AzureOpenAIEmbedding(
    model="text-embedding-3-small",
    deployment_name=os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
)

Settings.chunk_size    = 1024
Settings.chunk_overlap = 64

print("✅ Config ready")

## Step 3 — Load or Create Index

In [ ]:
def get_index():
    # Load existing index from disk
    if (STORAGE_DIR / "index_store.json").exists():
        print("Loading existing index...")
        storage_context = StorageContext.from_defaults(persist_dir=str(STORAGE_DIR))
        return load_index_from_storage(storage_context)

    # No index yet — check if data folder has files
    files = list(DATA_DIR.iterdir())
    if not files:
        print("⚠️  ./data is empty. Add your PDF/TXT files to the data folder first, then run this cell again.")
        return None

    # Build index from files
    print(f"Building index from {len(files)} file(s)...")
    docs  = SimpleDirectoryReader(str(DATA_DIR)).load_data()
    index = VectorStoreIndex.from_documents(docs, show_progress=True)
    index.storage_context.persist(persist_dir=str(STORAGE_DIR))
    print(f"✅ Index saved to {STORAGE_DIR}")
    return index

index = get_index()

## Step 4 — Query

In [ ]:
if index is None:
    print("No index yet. Add files to ./data and rebuild the index first.")
else:
    question = "What we do in day 2 in paris?"
    response = index.as_query_engine().query(question)
    print(response)

## Step 5 — Add a New Document

Put your file in `./data` and run this to add it to the existing index.

In [ ]:
# Change this to your actual file path
shutil.copy("../qna-quickstart-with-gpt-index/Kerala_Package.txt", DATA_DIR)

new_docs = SimpleDirectoryReader(str(DATA_DIR)).load_data()
for doc in new_docs:
    index.insert(doc)

index.storage_context.persist(persist_dir=str(STORAGE_DIR))
print("✅ Document added")

In [ ]:
if index is None:
    print("No index yet. Add files to ./data and rebuild the index first.")
else:
    question = "What we do in Munnar"
    response = index.as_query_engine().query(question)
    print(response)

## Step 6 — Delete a Document

In [ ]:
filename = "Kerala_Package.txt"   # change this

for doc_id, doc in list(index.docstore.docs.items()):
    if doc.metadata.get("file_name") == filename:
        index.delete_ref_doc(doc_id, delete_from_docstore=True)

index.storage_context.persist(persist_dir=str(STORAGE_DIR))
(DATA_DIR / filename).unlink(missing_ok=True)
print(f"✅ {filename} removed")

## Step 7 — Rebuild Index

Use after editing or replacing a file in `./data`.

In [ ]:
shutil.rmtree(str(STORAGE_DIR))
STORAGE_DIR.mkdir()

docs  = SimpleDirectoryReader(str(DATA_DIR)).load_data()
index = VectorStoreIndex.from_documents(docs, show_progress=True)
index.storage_context.persist(persist_dir=str(STORAGE_DIR))
print("✅ Index rebuilt")

In [ ]:
if index is None:
    print("No index yet. Add files to ./data and rebuild the index first.")
else:
    question = "What we do in Munnar"
    response = index.as_query_engine().query(question)
    print(response)

## Step 8 — Inspect Index

In [ ]:
from collections import defaultdict

file_map = defaultdict(int)
for doc in index.docstore.docs.values():
    fname = doc.metadata.get("file_name", "unknown")
    file_map[fname] += 1

print(f"Total chunks: {sum(file_map.values())}")
for fname, count in file_map.items():
    print(f"  {fname}  →  {count} chunks")